# LeetCode 731 — My Calendar II bằng Python

Link đề: https://leetcode.com/problems/my-calendar-ii/ · Độ khó: **Medium** · Chủ đề: Design, Segment Tree, Array, Binary Search, Ordered Set, Prefix Sum

Skeleton LeetCode:

```python
class MyCalendarTwo:

    def __init__(self):
        ...

    def book(self, startTime: int, endTime: int) -> bool:
        ...


# Your MyCalendarTwo object will be instantiated and called as such:
# obj = MyCalendarTwo()
# param_1 = obj.book(startTime,endTime)
```

Ghi chú: đây là bài **thiết kế lớp**. LeetCode gọi theo chuỗi lệnh (`["MyCalendarTwo","book","book",...]` kèm danh sách tham số) rồi so sánh danh sách kết quả (`null` ứng với `None`). Notebook tự dựng bộ chạy chuỗi lệnh `run_commands` để mô phỏng đúng cách LeetCode gọi, và chỉ dùng thư viện chuẩn (không có `SortedDict`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lịch cho phép **đặt trùng hai lần** (double booking) nhưng **cấm trùng ba lần** (triple booking); mỗi sự kiện là nửa khoảng `[start, end)`; `book` trả `True` và **lưu sự kiện** nếu không gây trùng ba, ngược lại trả `False` và **không lưu gì**.
2. Nắm các công cụ Python cho bài thiết kế lớp: `class`, `__init__`, `self`, thuộc tính của **thể hiện** (không phải thuộc tính lớp), danh sách các tuple, `any`/`all`, `dict.get`, `bisect.insort`, mảng hiệu (difference array), và cách viết bộ chạy chuỗi lệnh.
3. Đi từ **vét cạn đếm độ phủ tại các điểm đầu mút** (`O(n²)` mỗi lần đặt) → **hai danh sách: `booked` và `overlaps`** (`O(n)` mỗi lần) → **mảng hiệu trên trục thời gian thưa kèm hoàn tác** (`O(n)` mỗi lần) → **cây phân đoạn động (dynamic segment tree)** cộng khoảng + lấy max (`O(log C)` mỗi lần).
4. Hiểu mấu chốt: **trùng ba** nghĩa là có một thời điểm được phủ bởi `≥ 3` sự kiện; sự kiện mới `[s, e)` hợp lệ khi và chỉ khi nó **không giao với bất kỳ đoạn nào đã bị đặt trùng hai lần**. Hoặc nhìn qua độ phủ: `max độ phủ + 1 ≤ 2` trên `[s, e)`.
5. Tránh các lỗi kinh điển: dùng đoạn **đóng** thay vì nửa mở (`<=` thay vì `<`), thêm sự kiện **trước** khi kiểm tra rồi không hoàn tác, kiểm tra với `booked` thay vì `overlaps` (nhầm sang bài My Calendar I), lưu **cả đoạn mới** thay vì **phần giao** vào `overlaps`, và để danh sách ở **mức lớp** làm các thể hiện dùng chung dữ liệu.

## 2. Đề bài & ví dụ

Bạn đang cài đặt một chương trình lịch. Ta có thể thêm một sự kiện mới **nếu việc thêm không gây ra đặt trùng ba** (**triple booking**).

**Đặt trùng ba** xảy ra khi **ba sự kiện** có chung một giao **khác rỗng** (tức tồn tại một thời điểm thuộc cả ba sự kiện).

Một sự kiện là cặp số nguyên `startTime`, `endTime`, biểu diễn đặt lịch trên **nửa khoảng** `[startTime, endTime)`, tức các số thực `x` thoả `startTime <= x < endTime`.

Cài đặt lớp `MyCalendarTwo`:

- `MyCalendarTwo()` khởi tạo đối tượng lịch.
- `boolean book(int startTime, int endTime)` trả `true` nếu sự kiện có thể thêm vào lịch mà **không gây đặt trùng ba**; ngược lại trả `false` và **không thêm** sự kiện.

**Ví dụ 1**

```
Input
["MyCalendarTwo", "book", "book", "book", "book", "book", "book"]
[[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]
Output
[null, true, true, true, false, true, true]
```

Giải thích:

- `book(10, 20)` → `true`.
- `book(50, 60)` → `true`.
- `book(10, 40)` → `true` (trùng đôi với `[10, 20)`).
- `book(5, 15)` → `false`: `[10, 15)` đã được phủ bởi `[10,20)` và `[10,40)` rồi, thêm `[5,15)` thành trùng ba.
- `book(5, 10)` → `true`: không dùng thời điểm `10` vốn đã bị trùng đôi (nửa mở nên `10` không thuộc `[5,10)`).
- `book(25, 55)` → `true`: `[25, 40)` trùng đôi với sự kiện thứ ba, `[40, 50)` chỉ một lần, `[50, 55)` trùng đôi với sự kiện thứ hai.

**Ví dụ 2 (tự đặt, đối chiếu lại đề gốc nếu cần)**

```
["MyCalendarTwo", "book", "book", "book", "book"]
[[], [0, 10], [0, 10], [0, 10], [10, 20]]
→ [null, true, true, false, true]
```

Ba sự kiện y hệt: lần ba bị từ chối; `[10, 20)` **kề** (không giao) với `[0, 10)` nên được nhận.

**Ràng buộc:**

- `0 <= start < end <= 10^9`
- Tối đa `1000` lần gọi `book`.

Chú ý: toạ độ lên tới `10^9` nên **không thể** dùng mảng độ phủ theo từng đơn vị thời gian; ta cần cấu trúc thưa (danh sách đoạn, từ điển, cây phân đoạn động).

In [1]:
from typing import List, Optional
from bisect import bisect_left, insort
import random
import time


def run_commands(cls, ops, args):
    """Mô phỏng cách LeetCode gọi một bài thiết kế lớp:
    ops[i] là tên lớp (tạo đối tượng) hoặc tên phương thức; args[i] là danh sách tham số.
    Trả về danh sách kết quả; hàm khởi tạo ứng với None (null)."""
    obj, out = None, []
    for op, arg in zip(ops, args):
        if op == cls.__name__:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


def run_with(cls, ops, args):
    """Như run_commands nhưng lệnh đầu tiên luôn là khởi tạo, không cần trùng tên lớp
    (để chạy nhiều cài đặt khác tên với cùng một kịch bản)."""
    out = [None]
    obj = cls(*args[0])
    for op, arg in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*arg))
    return out


def script(bookings):
    """Tạo (ops, args) từ danh sách (start, end)."""
    return (["MyCalendarTwo"] + ["book"] * len(bookings),
            [[]] + [[s, e] for s, e in bookings])


ex1_ops = ["MyCalendarTwo", "book", "book", "book", "book", "book", "book"]
ex1_args = [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]
ex1_out = [None, True, True, True, False, True, True]

ex2_ops, ex2_args = script([(0, 10), (0, 10), (0, 10), (10, 20)])
ex2_out = [None, True, True, False, True]

examples = [
    (ex1_ops, ex1_args, ex1_out),
    (ex2_ops, ex2_args, ex2_out),          # ví dụ tự đặt
]
for ops, args, out in examples:
    print(args[1:], "-> kỳ vọng", out[1:])

[[10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]] -> kỳ vọng [True, True, True, False, True, True]
[[0, 10], [0, 10], [0, 10], [10, 20]] -> kỳ vọng [True, True, False, True]


## 3. Kiến thức nền

### 3.1. Lớp, `__init__`, `self` và trạng thái của đối tượng

Bài thiết kế yêu cầu ta viết **lớp** (`class`). `__init__` chạy một lần khi tạo đối tượng; bên trong ta gắn **trạng thái** vào đối tượng bằng `self.tên = giá_trị`. Mỗi phương thức nhận `self` để đọc/ghi trạng thái đó. **Mỗi đối tượng có bản trạng thái riêng.**

In [2]:
class Counter2:
    def __init__(self):
        self.count = 0              # trạng thái riêng của MỖI đối tượng

    def add(self, k):
        self.count += k
        return self.count

a, b = Counter2(), Counter2()
a.add(5)
b.add(1)
print(a.count, b.count)             # 5 1 : hai đối tượng độc lập

5 1


### 3.2. Nửa khoảng `[s, e)` và phép giao

Nửa khoảng `[s, e)` gồm mọi `x` với `s <= x < e`: **gồm đầu trái, không gồm đầu phải**. Hai nửa khoảng `[a, b)` và `[c, d)`:

- **Giao nhau** (có điểm chung) khi `max(a, c) < min(b, d)`. Dấu `<` **chặt**: nếu `max(a, c) == min(b, d)` thì chúng chỉ **kề nhau**.
- **Phần giao** là `[max(a, c), min(b, d))`.

Ví dụ `[10, 20)` và `[20, 30)` **không giao nhau** (kề nhau), `[10, 20)` và `[15, 30)` giao nhau tại `[15, 20)`.

In [3]:
def intersect(a, b):
    lo, hi = max(a[0], b[0]), min(a[1], b[1])
    return (lo, hi) if lo < hi else None

print(intersect((10, 20), (20, 30)))     # None: chỉ kề nhau
print(intersect((10, 20), (15, 30)))     # (15, 20)
print(intersect((10, 40), (5, 15)))      # (10, 15)
print(intersect((5, 10), (10, 20)))      # None

None
(15, 20)
(10, 15)
None


### 3.3. Danh sách tuple, `any`, vòng lặp có điều kiện

Ta lưu sự kiện dưới dạng **tuple** `(start, end)` trong list. `any(điều_kiện for x in dãy)` trả `True` nếu **có ít nhất một** phần tử thoả điều kiện (dừng sớm khi gặp), `all(...)` trả `True` nếu **mọi** phần tử thoả.

In [4]:
events = [(10, 20), (50, 60), (10, 40)]
print(any(a < 15 < b for a, b in events))       # True: 15 nằm trong [10, 20)
print(all(b - a >= 10 for a, b in events))
print(sum(1 for a, b in events if a <= 12 < b)) # độ phủ tại thời điểm 12 là 2

True
True
2


### 3.4. `dict.get`, `bisect.insort` và duyệt theo thứ tự khoá

Để thay `SortedDict` (không có trong thư viện chuẩn) ta dùng một `dict` cho **giá trị** và một **list đã sắp xếp** cho **khoá**, giữ sắp xếp bằng `bisect.insort` (chèn vào đúng vị trí, `O(n)` do dịch chuyển phần tử). `d.get(k, 0)` trả `d[k]` hoặc `0` nếu chưa có khoá.

In [5]:
diff, keys = {}, []
def add_point(t, delta):
    if t not in diff:
        insort(keys, t)                  # giữ keys luôn tăng dần
    diff[t] = diff.get(t, 0) + delta

for t, d in [(20, -1), (10, +1), (10, +1), (40, -1)]:
    add_point(t, d)
print(keys, [diff[k] for k in keys])
print(bisect_left(keys, 20))             # vị trí của khoá 20 trong keys

[10, 20, 40] [2, -1, -1]
1


### 3.5. Mảng hiệu (difference array) và tổng tiền tố

Muốn biết **mỗi thời điểm được phủ bao nhiêu sự kiện** mà không lưu từng thời điểm: với sự kiện `[s, e)`, ghi **`+1` tại `s`** và **`−1` tại `e`**. Duyệt các mốc theo thứ tự tăng dần và cộng dồn (tổng tiền tố): giá trị cộng dồn tại một mốc chính là độ phủ **từ mốc đó đến mốc kế tiếp**.

Ví dụ ba sự kiện `[10,20)`, `[50,60)`, `[10,40)`: các mốc `10:+2, 20:−1, 40:−1, 50:+1, 60:−1`; cộng dồn: `10→2, 20→1, 40→0, 50→1, 60→0`.

In [6]:
points = {}
for s, e in [(10, 20), (50, 60), (10, 40)]:
    points[s] = points.get(s, 0) + 1
    points[e] = points.get(e, 0) - 1
cur = 0
for t in sorted(points):
    cur += points[t]
    print(f"từ mốc {t:>2} độ phủ = {cur}")

từ mốc 10 độ phủ = 2
từ mốc 20 độ phủ = 1
từ mốc 40 độ phủ = 0
từ mốc 50 độ phủ = 1
từ mốc 60 độ phủ = 0


### 3.6. Rollback (hoàn tác) khi thử rồi huỷ

Nhiều lời giải **thêm thử** sự kiện vào cấu trúc, kiểm tra điều kiện, rồi nếu vi phạm thì **hoàn tác** (cộng/trừ ngược lại). Nếu quên hoàn tác thì sự kiện bị **từ chối** vẫn "ám" vào dữ liệu — một lỗi rất phổ biến (xem mục 13). Cách khác là **kiểm tra trước, chỉ ghi khi hợp lệ**, không cần hoàn tác.

In [7]:
state = [0, 0, 0, 0]
def try_add(i, limit):
    state[i] += 1                       # thêm thử
    if state[i] > limit:
        state[i] -= 1                   # hoàn tác khi vi phạm
        return False
    return True

print([try_add(0, 2) for _ in range(4)], state)   # [True, True, False, False] [2, 0, 0, 0]

[True, True, False, False] [2, 0, 0, 0]


### 3.7. Bộ chạy chuỗi lệnh

`run_commands` (đã định nghĩa ở mục 2) duyệt song song `ops` và `args`. Với lệnh đầu tiên (tên lớp) nó tạo đối tượng và ghi `None`; với các lệnh sau, `getattr(obj, op)(*arg)` gọi phương thức tên `op` với các tham số `arg` (dấu `*` "bung" list thành đối số riêng). Ta thử với một lớp đồ chơi:

In [8]:
class Toy:
    def __init__(self):
        self.total = 0
    def book(self, s, e):
        self.total += e - s
        return self.total < 30

print(run_commands(Toy, ["Toy", "book", "book", "book"], [[], [0, 10], [0, 10], [0, 10]]))

[None, True, True, False]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Theo dõi ví dụ 1 bằng mắt, vẽ trục thời gian và độ phủ sau mỗi lần đặt thành công:

```
book(10,20):  [10,20)=1
book(50,60):  [10,20)=1, [50,60)=1
book(10,40):  [10,20)=2, [20,40)=1, [50,60)=1
book(5,15):   thử: [10,15) sẽ lên 3 -> TỪ CHỐI (không ghi)
book(5,10):   [5,10)=1  (không chạm [10,20)=2)  -> nhận
book(25,55):  [25,40)=2, [40,50)=1, [50,55)=2  -> nhận (không đoạn nào lên 3)
```

Quan sát: ta chỉ quan tâm các vùng **đã được phủ 2 lần** — gọi là vùng **trùng đôi**. Sự kiện mới hợp lệ khi **không giao** với vùng trùng đôi nào. Ở bước `book(5, 15)` vùng trùng đôi là `[10, 20)` (từ `[10,20)` và `[10,40)`), và `[5,15)` giao với nó.

Và khi nhận một sự kiện mới, **vùng trùng đôi mới** sinh ra chính là phần giao của nó với từng sự kiện đã có (vì sự kiện mới không đụng vùng trùng đôi nên phần giao với mỗi sự kiện cũ chỉ tạo độ phủ 2, không phải 3).

Vậy ta cần lưu hai thứ: danh sách mọi sự kiện đã nhận (`booked`) và danh sách các đoạn trùng đôi (`overlaps`). Đó là cách 2.

In [9]:
booked, overlaps = [], []
for s, e in [(10, 20), (50, 60), (10, 40)]:
    for a, b in booked:
        x = intersect((a, b), (s, e))
        if x:
            overlaps.append(x)
    booked.append((s, e))
    print(f"sau book({s},{e}): booked={booked} overlaps={overlaps}")
print("book(5,15) giao với vùng trùng đôi?", any(intersect(o, (5, 15)) for o in overlaps))
print("book(5,10) giao với vùng trùng đôi?", any(intersect(o, (5, 10)) for o in overlaps))

sau book(10,20): booked=[(10, 20)] overlaps=[]
sau book(50,60): booked=[(10, 20), (50, 60)] overlaps=[]
sau book(10,40): booked=[(10, 20), (50, 60), (10, 40)] overlaps=[(10, 20)]
book(5,15) giao với vùng trùng đôi? True
book(5,10) giao với vùng trùng đôi? False


## 5. Cách 1 — Brute Force / baseline: đếm độ phủ tại các điểm đầu mút

**Ý tưởng "dịch nguyên văn đề bài":** sự kiện mới `[s, e)` gây trùng ba nếu có thời điểm `x ∈ [s, e)` mà **đã có ít nhất 2 sự kiện** phủ `x`.

Không cần thử mọi `x` (có tới `10^9`). Độ phủ chỉ **tăng** khi đi qua một điểm **bắt đầu** của sự kiện, nên giá trị lớn nhất trên `[s, e)` đạt được tại `x = s` hoặc tại một điểm `a` **bắt đầu** của sự kiện cũ với `s <= a < e`. Vậy:

1. Tập điểm ứng viên: `{s}` ∪ `{a : (a, b) đã có, s <= a < e}`.
2. Với mỗi ứng viên `x`: đếm số sự kiện cũ phủ `x` (tức `a <= x < b`). Nếu `>= 2` → từ chối.
3. Nếu qua hết → ghi sự kiện, trả `True`.

In [10]:
class CalBrute:
    def __init__(self):
        self.events = []

    def book(self, startTime: int, endTime: int) -> bool:
        s, e = startTime, endTime
        candidates = {s}
        for a, b in self.events:
            if s <= a < e:
                candidates.add(a)
        for x in candidates:
            cover = sum(1 for a, b in self.events if a <= x < b)
            if cover >= 2:
                return False
        self.events.append((s, e))
        return True

for ops, args, expected in examples:
    got = run_with(CalBrute, ops, args)
    print(got)
    assert got == expected

[None, True, True, True, False, True, True]
[None, True, True, False, True]


**Độ phức tạp.** Mỗi lần `book` có tối đa `n` ứng viên, mỗi ứng viên đếm `O(n)` sự kiện.

- **Time:** `O(n²)` mỗi lần `book`, tức `O(n³)` cho `n` lần gọi (với `n = 1000` có thể tới `~10^9` phép xấu nhất, nhưng thực tế ít ứng viên hơn).
- **Space:** `O(n)` lưu sự kiện.

**Nút thắt:** ta **đếm lại từ đầu** độ phủ tại mỗi ứng viên. Nếu **lưu sẵn** các vùng đã trùng đôi, mỗi lần chỉ cần so với vùng đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** thay vì đếm độ phủ, hãy lưu **trực tiếp các vùng đã bị phủ đúng 2 lần** (`overlaps`). Khi đó:

- **Kiểm tra** sự kiện mới `[s, e)`: nó hợp lệ **khi và chỉ khi không giao với bất kỳ đoạn nào trong `overlaps`** (vì giao với vùng trùng đôi nghĩa là có điểm phủ ≥ 3). Chỉ `O(|overlaps|)`.
- **Cập nhật** khi nhận: với mỗi sự kiện cũ `(a, b)` trong `booked`, phần giao `[max(a,s), min(b,e))` (nếu khác rỗng) là một vùng **vừa thành trùng đôi** → thêm vào `overlaps`. Vì `[s,e)` không đụng vùng trùng đôi cũ nên các phần giao này đều chỉ có độ phủ đúng `2`. Rồi thêm `(s, e)` vào `booked`.

Hai bước đều `O(n)` mỗi lần gọi, và **chỉ ghi khi hợp lệ** nên không cần hoàn tác.

Một hướng khác: nhìn độ phủ như một **hàm bậc thang** trên trục thời gian và cập nhật bằng **mảng hiệu** (cách 3), hoặc dùng **cây phân đoạn** để cộng `+1` cho một khoảng và hỏi **max** trên một khoảng (cách 4). Ta lần lượt cài đặt.

In [11]:
# Thử bước đột phá trên ví dụ 1
booked, overlaps = [], []
def book_try(s, e):
    if any(max(a, s) < min(b, e) for a, b in overlaps):      # giao với vùng trùng đôi?
        return False
    for a, b in booked:
        lo, hi = max(a, s), min(b, e)
        if lo < hi:
            overlaps.append((lo, hi))
    booked.append((s, e))
    return True

print([book_try(s, e) for s, e in [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]])
print("overlaps =", overlaps)

[True, True, True, False, True, True]
overlaps = [(10, 20), (50, 55), (25, 40)]


## 7. Các cách tiếp theo

### Cách 2 — Hai danh sách `booked` và `overlaps` (`O(n)` mỗi lần)

**Giải thích từng đoạn:**

1. `__init__`: tạo `self.booked = []` (mọi sự kiện đã nhận) và `self.overlaps = []` (các đoạn đang trùng đôi). Gắn vào `self` để **mỗi đối tượng có danh sách riêng**.
2. `book(s, e)`: nếu tồn tại `(a, b)` trong `overlaps` với `max(a, s) < min(b, e)` → trả `False` (không ghi gì).
3. Ngược lại: với mỗi `(a, b)` trong `booked`, nếu `max(a, s) < min(b, e)` thì thêm `(max(a, s), min(b, e))` vào `overlaps`.
4. Thêm `(s, e)` vào `booked`; trả `True`.

In [12]:
class CalTwoLists:
    def __init__(self):
        self.booked = []
        self.overlaps = []

    def book(self, startTime: int, endTime: int) -> bool:
        s, e = startTime, endTime
        for a, b in self.overlaps:
            if max(a, s) < min(b, e):
                return False
        for a, b in self.booked:
            lo, hi = max(a, s), min(b, e)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((s, e))
        return True

for ops, args, expected in examples:
    got = run_with(CalTwoLists, ops, args)
    print(got)
    assert got == expected

[None, True, True, True, False, True, True]
[None, True, True, False, True]


In [13]:
class MyCalendarTwo:

    def __init__(self):
        self.booked = []
        self.overlaps = []

    def book(self, startTime: int, endTime: int) -> bool:
        for a, b in self.overlaps:
            if max(a, startTime) < min(b, endTime):
                return False
        for a, b in self.booked:
            lo, hi = max(a, startTime), min(b, endTime)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((startTime, endTime))
        return True


for ops, args, expected in examples:
    assert run_commands(MyCalendarTwo, ops, args) == expected
print("OK")

OK


**Độ phức tạp (cách 2):**

- **Time:** mỗi `book` duyệt `overlaps` và `booked`: `O(n)`; tổng `O(n²)` cho `n` lần gọi (`n = 1000` → `~10^6`).
- **Space:** `O(n)`: `booked` có tối đa `n` đoạn, `overlaps` cũng có tối đa `n` đoạn (mỗi lần nhận thêm nhiều nhất `|booked|` đoạn nhưng chúng không chồng lên nhau quá 2 lần, nên tổng vẫn `O(n)`).

Thực tế `overlaps` có thể lớn lên tới bậc `O(n)` đoạn vì mỗi sự kiện cũ cho tối đa một phần giao với sự kiện mới, nhưng nhiều đoạn đó **rời nhau** (vì độ phủ tối đa là 2 nên hai phần giao khác nhau của cùng một sự kiện mới không thể chồng lên nhau).

### Cách 3 — Mảng hiệu trên trục thời gian thưa + hoàn tác (`O(n)` mỗi lần)

**Ý tưởng khác hẳn:** không lưu các đoạn, mà lưu **hàm độ phủ** dưới dạng mảng hiệu thưa: `diff[t]` = thay đổi độ phủ tại mốc `t`.

**Giải thích từng đoạn:**

1. `self.diff` là `dict` mốc → giá trị; `self.keys` là list các mốc **luôn sắp tăng** (dùng `insort`).
2. `book(s, e)`: **thêm thử** `+1` tại `s`, `−1` tại `e`.
3. Duyệt các mốc tăng dần, cộng dồn `cur += diff[k]`; nếu `cur > 2` tại một mốc nào đó → **hoàn tác** (`−1` tại `s`, `+1` tại `e`) và trả `False`.
4. Nếu không vi phạm → giữ nguyên, trả `True`.

Sự kiện nửa mở `[s, e)`: `+1` ở `s` và `−1` ở `e` nên điểm `e` **không** được tính là bị phủ — đúng với đề (xem ví dụ `book(5, 10)`).

In [14]:
class CalDiff:
    def __init__(self):
        self.diff = {}
        self.keys = []

    def _add(self, t, d):
        if t not in self.diff:
            insort(self.keys, t)
            self.diff[t] = 0
        self.diff[t] += d

    def book(self, startTime: int, endTime: int) -> bool:
        self._add(startTime, 1)
        self._add(endTime, -1)
        cur = 0
        for k in self.keys:
            cur += self.diff[k]
            if cur > 2:
                self._add(startTime, -1)          # hoàn tác
                self._add(endTime, 1)
                return False
        return True

for ops, args, expected in examples:
    got = run_with(CalDiff, ops, args)
    print(got)
    assert got == expected

[None, True, True, True, False, True, True]
[None, True, True, False, True]


In [15]:
class MyCalendarTwo:

    def __init__(self):
        self.diff = {}
        self.keys = []

    def _add(self, t, d):
        if t not in self.diff:
            insort(self.keys, t)
            self.diff[t] = 0
        self.diff[t] += d

    def book(self, startTime: int, endTime: int) -> bool:
        self._add(startTime, 1)
        self._add(endTime, -1)
        cur = 0
        for k in self.keys:
            cur += self.diff[k]
            if cur > 2:
                self._add(startTime, -1)
                self._add(endTime, 1)
                return False
        return True


for ops, args, expected in examples:
    assert run_commands(MyCalendarTwo, ops, args) == expected
print("OK")

OK


**Độ phức tạp (cách 3):**

- **Time:** mỗi `book`: hai lần `insort` (`O(n)` do dịch chuyển, nhưng với `n ≤ 2000` rất nhanh) và một lượt duyệt `O(n)`; tổng `O(n²)`.
- **Space:** `O(n)` (tối đa `2n` mốc; mốc đã hoàn tác vẫn nằm trong `keys` với giá trị `0` nhưng không ảnh hưởng đúng sai).

Ưu điểm: **rất ngắn**, và dễ mở rộng sang *My Calendar III* (độ phủ lớn nhất) hay *Meeting Rooms II*.

### Cách 4 — Cây phân đoạn động: cộng `+1` cho khoảng, hỏi max (`O(log C)` mỗi lần)

**Ý tưởng:** coi trục thời gian là mảng độ phủ `cover[0 .. 10^9 − 1]` (chỉ **tạo nút khi cần**: *dynamic segment tree*). Sự kiện `[s, e)` tương ứng khoảng số nguyên `[s, e − 1]`.

- **Hỏi:** `query(s, e − 1)` = `max` của `cover` trên khoảng. Nếu `>= 2` → thêm sự kiện sẽ lên 3 → từ chối.
- **Ghi:** nếu hợp lệ, `update(s, e − 1)` cộng `+1` cho cả khoảng.

**Giải thích cấu trúc:** mỗi nút lưu `add` (số lần cộng **trọn nút**, không đẩy xuống con) và `mx` (độ phủ lớn nhất trong đoạn của nút, **đã tính** `add` của chính nó). Khi cập nhật khoảng phủ **trọn** đoạn của nút → `add += 1`, `mx += 1`. Nếu chỉ phủ một phần → đệ quy xuống con (tạo con khi chưa có) rồi `mx = add + max(mx[con trái], mx[con phải])`. Khi hỏi: nếu phủ trọn → trả `mx`; nếu không thì lấy max từ các con rồi **cộng** `add` của nút. Các nút chưa tạo coi như `0`.

Vì `C = 10^9 ≈ 2^30` nên độ sâu `≈ 30`; mỗi thao tác chạm `O(log C)` nút.

In [16]:
class CalSegTree:
    HI = 10 ** 9 - 1                    # chỉ số thời điểm lớn nhất: [0, 10^9 - 1]

    def __init__(self):
        self.lc, self.rc, self.mx, self.add = [-1], [-1], [0], [0]     # nút 0 là gốc

    def _new(self):
        self.lc.append(-1); self.rc.append(-1); self.mx.append(0); self.add.append(0)
        return len(self.mx) - 1

    def _query(self, node, l, r, ql, qr):
        if node == -1:
            return 0
        if ql <= l and r <= qr:
            return self.mx[node]
        mid = (l + r) // 2
        res = 0
        if ql <= mid:
            res = max(res, self._query(self.lc[node], l, mid, ql, qr))
        if qr > mid:
            res = max(res, self._query(self.rc[node], mid + 1, r, ql, qr))
        return res + self.add[node]

    def _update(self, node, l, r, ql, qr):
        if ql <= l and r <= qr:
            self.mx[node] += 1
            self.add[node] += 1
            return
        mid = (l + r) // 2
        if ql <= mid:
            if self.lc[node] == -1:
                self.lc[node] = self._new()
            self._update(self.lc[node], l, mid, ql, qr)
        if qr > mid:
            if self.rc[node] == -1:
                self.rc[node] = self._new()
            self._update(self.rc[node], mid + 1, r, ql, qr)
        left = self.mx[self.lc[node]] if self.lc[node] != -1 else 0
        right = self.mx[self.rc[node]] if self.rc[node] != -1 else 0
        self.mx[node] = self.add[node] + max(left, right)

    def book(self, startTime: int, endTime: int) -> bool:
        if self._query(0, 0, self.HI, startTime, endTime - 1) >= 2:
            return False
        self._update(0, 0, self.HI, startTime, endTime - 1)
        return True

for ops, args, expected in examples:
    got = run_with(CalSegTree, ops, args)
    print(got)
    assert got == expected

[None, True, True, True, False, True, True]
[None, True, True, False, True]


In [17]:
class MyCalendarTwo:
    HI = 10 ** 9 - 1

    def __init__(self):
        self.lc, self.rc, self.mx, self.add = [-1], [-1], [0], [0]

    def _new(self):
        self.lc.append(-1); self.rc.append(-1); self.mx.append(0); self.add.append(0)
        return len(self.mx) - 1

    def _query(self, node, l, r, ql, qr):
        if node == -1:
            return 0
        if ql <= l and r <= qr:
            return self.mx[node]
        mid = (l + r) // 2
        res = 0
        if ql <= mid:
            res = max(res, self._query(self.lc[node], l, mid, ql, qr))
        if qr > mid:
            res = max(res, self._query(self.rc[node], mid + 1, r, ql, qr))
        return res + self.add[node]

    def _update(self, node, l, r, ql, qr):
        if ql <= l and r <= qr:
            self.mx[node] += 1
            self.add[node] += 1
            return
        mid = (l + r) // 2
        if ql <= mid:
            if self.lc[node] == -1:
                self.lc[node] = self._new()
            self._update(self.lc[node], l, mid, ql, qr)
        if qr > mid:
            if self.rc[node] == -1:
                self.rc[node] = self._new()
            self._update(self.rc[node], mid + 1, r, ql, qr)
        left = self.mx[self.lc[node]] if self.lc[node] != -1 else 0
        right = self.mx[self.rc[node]] if self.rc[node] != -1 else 0
        self.mx[node] = self.add[node] + max(left, right)

    def book(self, startTime: int, endTime: int) -> bool:
        if self._query(0, 0, self.HI, startTime, endTime - 1) >= 2:
            return False
        self._update(0, 0, self.HI, startTime, endTime - 1)
        return True


for ops, args, expected in examples:
    assert run_commands(MyCalendarTwo, ops, args) == expected
print("OK")

OK


**Độ phức tạp (cách 4):**

- **Time:** mỗi `book` gồm một `query` và tối đa một `update`, mỗi cái `O(log C)` với `C = 10^9` (~30 tầng, thực tế vài chục đến vài trăm nút chạm); tổng `O(n log C)`.
- **Space:** `O(n log C)` nút (mỗi lần cập nhật tạo thêm tối đa `~2 log C` nút mới).

Cách này **tối ưu về lý thuyết** và là mẫu cho các bài "cộng khoảng, hỏi max trên khoảng" với toạ độ lớn. Với `n ≤ 1000` cả cách 2 và 3 đã đủ nhanh.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (`booked` / `overlaps`) trên ví dụ 1

| Lần | `book(s, e)` | Giao với `overlaps`? | Phần giao mới thêm vào `overlaps` | `booked` sau | Kết quả |
|---|---|---|---|---|---|
| 1 | `(10, 20)` | không (rỗng) | — | `[(10,20)]` | `True` |
| 2 | `(50, 60)` | không | `(10,20)∩(50,60)` rỗng | `[(10,20),(50,60)]` | `True` |
| 3 | `(10, 40)` | không | `(10,20)` | `+(10,40)` | `True` |
| 4 | `(5, 15)` | **có**: `(10,20)` → `max(10,5)=10 < min(20,15)=15` | — (không ghi) | không đổi | `False` |
| 5 | `(5, 10)` | `max(10,5)=10 < min(20,10)=10`? **không** (chỉ kề) | `(5,10)∩(10,20)` rỗng, `∩(50,60)` rỗng, `∩(10,40)` rỗng | `+(5,10)` | `True` |
| 6 | `(25, 55)` | `(10,20)`: `25 < 20`? không → không giao | `(10,40)`→`(25,40)`; `(50,60)`→`(50,55)` | `+(25,55)` | `True` |

Hàm `trace_two_lists` in đúng các bước đó.

In [18]:
def trace_two_lists(bookings):
    booked, overlaps = [], []
    for step, (s, e) in enumerate(bookings, 1):
        hit = [o for o in overlaps if max(o[0], s) < min(o[1], e)]
        if hit:
            print(f"lần {step}: book({s},{e}) -> False | giao vùng trùng đôi {hit} | không ghi gì")
            continue
        new = []
        for a, b in booked:
            lo, hi = max(a, s), min(b, e)
            if lo < hi:
                new.append((lo, hi))
        overlaps.extend(new)
        booked.append((s, e))
        print(f"lần {step}: book({s},{e}) -> True  | phần giao mới {new} | booked={booked} | overlaps={overlaps}")

trace_two_lists([(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)])

lần 1: book(10,20) -> True  | phần giao mới [] | booked=[(10, 20)] | overlaps=[]
lần 2: book(50,60) -> True  | phần giao mới [] | booked=[(10, 20), (50, 60)] | overlaps=[]
lần 3: book(10,40) -> True  | phần giao mới [(10, 20)] | booked=[(10, 20), (50, 60), (10, 40)] | overlaps=[(10, 20)]
lần 4: book(5,15) -> False | giao vùng trùng đôi [(10, 20)] | không ghi gì
lần 5: book(5,10) -> True  | phần giao mới [] | booked=[(10, 20), (50, 60), (10, 40), (5, 10)] | overlaps=[(10, 20)]
lần 6: book(25,55) -> True  | phần giao mới [(50, 55), (25, 40)] | booked=[(10, 20), (50, 60), (10, 40), (5, 10), (25, 55)] | overlaps=[(10, 20), (50, 55), (25, 40)]


### 8.2. Trace cách 3 (mảng hiệu) tại lần `book(5, 15)` bị từ chối

Trước lần này các mốc là `10:+2, 20:−1, 40:−1, 50:+1, 60:−1`. Thêm thử `+1` tại `5` và `−1` tại `15` rồi cộng dồn theo mốc tăng dần:

| Mốc | `diff` | Cộng dồn |
|---|---|---|
| 5 | +1 | 1 |
| 10 | +2 | **3** → vượt 2 → hoàn tác, trả `False` |

Ở lần `book(5, 10)` sau đó: mốc `5:+1, 10:+2−1=+1, ...` (vì thêm `−1` tại `10`): cộng dồn `1, 2, 2, 1, 0, 1, 0` (các mốc `5, 10, 15, 20, 40, 50, 60`; mốc `15` có `diff = 0` do đã hoàn tác) — không vượt 2, nhận.

In [19]:
def trace_diff(bookings):
    diff, keys = {}, []
    def add(t, d):
        if t not in diff:
            insort(keys, t)
            diff[t] = 0
        diff[t] += d
    for step, (s, e) in enumerate(bookings, 1):
        add(s, 1); add(e, -1)
        cur, line, ok = 0, [], True
        for k in keys:
            cur += diff[k]
            line.append(f"{k}:{cur}")
            if cur > 2:
                ok = False
                break
        if not ok:
            add(s, -1); add(e, 1)
        print(f"lần {step}: book({s},{e}) cộng dồn [{', '.join(line)}] -> {ok}")

trace_diff([(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)])

lần 1: book(10,20) cộng dồn [10:1, 20:0] -> True
lần 2: book(50,60) cộng dồn [10:1, 20:0, 50:1, 60:0] -> True
lần 3: book(10,40) cộng dồn [10:2, 20:1, 40:0, 50:1, 60:0] -> True
lần 4: book(5,15) cộng dồn [5:1, 10:3] -> False
lần 5: book(5,10) cộng dồn [5:1, 10:2, 15:2, 20:1, 40:0, 50:1, 60:0] -> True
lần 6: book(25,55) cộng dồn [5:1, 10:2, 15:2, 20:1, 25:2, 40:1, 50:2, 55:1, 60:0] -> True


### 8.3. Trace cách 4 (cây phân đoạn) — số nút và `query`

Ta không in từng nút (quá nhiều), mà in kết quả `query` (độ phủ lớn nhất trên khoảng) và số nút đã tạo sau mỗi lần.

In [20]:
def trace_segtree(bookings):
    cal = CalSegTree()
    for step, (s, e) in enumerate(bookings, 1):
        q = cal._query(0, 0, cal.HI, s, e - 1)
        ok = cal.book(s, e)
        print(f"lần {step}: book({s},{e}) max độ phủ trên [{s},{e-1}] trước khi thêm = {q} -> {ok} | số nút = {len(cal.mx)}")

trace_segtree([(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)])

lần 1: book(10,20) max độ phủ trên [10,19] trước khi thêm = 0 -> True | số nút = 37
lần 2: book(50,60) max độ phủ trên [50,59] trước khi thêm = 0 -> True | số nút = 45
lần 3: book(10,40) max độ phủ trên [10,39] trước khi thêm = 1 -> True | số nút = 50
lần 4: book(5,15) max độ phủ trên [5,14] trước khi thêm = 2 -> False | số nút = 50
lần 5: book(5,10) max độ phủ trên [5,9] trước khi thêm = 0 -> True | số nút = 56
lần 6: book(25,55) max độ phủ trên [25,54] trước khi thêm = 1 -> True | số nút = 62


## 9. Code nộp LeetCode

Đây là cách 2 (hai danh sách) — ngắn, dễ nhớ, `O(n)` mỗi lần, đủ nhanh cho `n ≤ 1000`. Khi nộp lên LeetCode chỉ cần lớp `MyCalendarTwo`.

In [21]:
class MyCalendarTwo:

    def __init__(self):
        self.booked = []        # mọi sự kiện đã nhận
        self.overlaps = []      # các đoạn đã bị phủ đúng 2 lần

    def book(self, startTime: int, endTime: int) -> bool:
        for a, b in self.overlaps:
            if max(a, startTime) < min(b, endTime):
                return False
        for a, b in self.booked:
            lo, hi = max(a, startTime), min(b, endTime)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((startTime, endTime))
        return True


# Your MyCalendarTwo object will be instantiated and called as such:
# obj = MyCalendarTwo()
# param_1 = obj.book(startTime,endTime)

for ops, args, expected in examples:
    got = run_commands(MyCalendarTwo, ops, args)
    print(got)
    assert got == expected
print("Chạy ví dụ: OK")

[None, True, True, True, False, True, True]
[None, True, True, False, True]
Chạy ví dụ: OK


## 10. Vì sao đúng

**Bất biến (cách 2).** Sau mỗi lần `book` thành công:

1. `booked` chứa mọi sự kiện đã nhận.
2. Tập hợp các điểm có **độ phủ đúng 2** bằng hợp của các đoạn trong `overlaps`. Và **không có điểm nào độ phủ ≥ 3**.

**Chứng minh quy nạp.**

- Cơ sở: lịch rỗng, hai tập rỗng.
- Bước: xét sự kiện mới `E = [s, e)`.
  - Nếu `E` giao một đoạn trong `overlaps` ở điểm `x`: `x` đã phủ 2 lần, thêm `E` thành 3 → **phải từ chối**. Cách 2 trả `False` và **không đổi** gì → bất biến giữ nguyên.
  - Nếu `E` không giao đoạn nào trong `overlaps`: với mọi `x ∈ E`, độ phủ hiện tại `≤ 1`, nên sau khi thêm `≤ 2` → không trùng ba, **được nhận**. Các điểm có độ phủ mới đúng `2` trong `E` chính là các điểm `x ∈ E` đã có độ phủ `1`, tức `x ∈ E ∩ (a, b)` với đúng một sự kiện cũ `(a, b)` (nếu thuộc hai sự kiện cũ thì độ phủ đã `2` và `E` đã giao `overlaps`). Vì vậy hợp của các phần giao `E ∩ (a, b)` trên mọi sự kiện cũ **chính là** tập các điểm mới có độ phủ 2 trong `E`. Cách 2 thêm đúng các phần giao đó vào `overlaps` → bất biến giữ.

**Cách 3 và 4** là hai cách biểu diễn khác của **hàm độ phủ**: mảng hiệu (cộng dồn theo mốc) và cây phân đoạn (max trên khoảng). Điều kiện từ chối là "tồn tại điểm có độ phủ sau khi thêm `> 2`", tương đương "độ phủ lớn nhất trên `[s, e)` hiện tại `≥ 2`".

Ta kiểm chứng bằng code: 4 cài đặt cho **cùng kết quả** với một **oracle độc lập** đếm độ phủ từng đơn vị thời gian trên trục nhỏ.

In [22]:
class CalUnitOracle:
    """Oracle độc lập: mảng độ phủ theo từng đơn vị thời gian (chỉ dùng được với toạ độ nhỏ)."""
    def __init__(self, size=64):
        self.cnt = [0] * size

    def book(self, s, e):
        if max(self.cnt[s:e]) >= 2:
            return False
        for x in range(s, e):
            self.cnt[x] += 1
        return True

IMPLS = {
    "brute": CalBrute,
    "two_lists": CalTwoLists,
    "diff": CalDiff,
    "segtree": CalSegTree,
}

rng = random.Random(731)
count = 0
for _ in range(300):
    calls = rng.randint(1, 40)
    bookings = []
    for _ in range(calls):
        s = rng.randint(0, 55)
        e = rng.randint(s + 1, min(s + rng.choice([2, 5, 15]), 60))
        bookings.append((s, e))
    ops, args = script(bookings)
    oracle = run_with(CalUnitOracle, ops, args)
    for name, cls in IMPLS.items():
        assert run_with(cls, ops, args) == oracle, (name, bookings)
    count += 1
print("đã so", count, "kịch bản ngẫu nhiên nhỏ giữa 4 cài đặt và oracle theo đơn vị: OK")

đã so

 300 kịch bản ngẫu nhiên nhỏ giữa 4 cài đặt và oracle theo đơn vị: OK


## 11. Time / Space Complexity

Gọi `n` là số lần gọi `book` (≤ 1000) và `C = 10^9` là biên toạ độ.

| Cách | Time mỗi `book` | Time tổng | Space | Ghi chú |
|---|---|---|---|---|
| 1. Đếm tại đầu mút | `O(n²)` | `O(n³)` | `O(n)` | dùng làm oracle |
| 2. `booked` + `overlaps` | `O(n)` | `O(n²)` | `O(n)` | **cách nộp chuẩn** |
| 3. Mảng hiệu thưa + hoàn tác | `O(n)` | `O(n²)` | `O(n)` | rất ngắn, mở rộng được |
| 4. Cây phân đoạn động | `O(log C)` | `O(n log C)` | `O(n log C)` | nhanh nhất về lý thuyết |

Với `n = 1000`: cách 2 và 3 cỡ `10^6` phép — chạy chớp mắt. Cách 4 chỉ thực sự cần khi `n` lên tới hàng trăm nghìn.

## 12. Bộ test

Gồm: ví dụ đề (qua `run_commands`, mô phỏng đúng cách LeetCode gọi), edge case (đoạn kề nhau, ba đoạn trùng hệt nhau, đoạn lồng nhau, đoạn dài `[0, 10^9)`, từ chối **không** làm thay đổi trạng thái), phản ví dụ cho cách làm sai, đối chiếu **4 cài đặt với nhau** và với **oracle độc lập** trên nhiều kịch bản ngẫu nhiên có seed (đúng ràng buộc), và test lớn `1000` lệnh.

In [23]:
def check_script(bookings, expected=None, impls=IMPLS):
    ops, args = script(bookings)
    ref = None
    for name, cls in impls.items():
        got = run_with(cls, ops, args)
        if expected is not None:
            assert got == [None] + expected, (name, bookings, got)
        if ref is None:
            ref = got
        assert got == ref, (name, bookings)
    assert run_commands(MyCalendarTwo, ops, args) == ref        # lớp nộp bài (cách 2)
    return ref[1:]

# ví dụ đề + ví dụ tự đặt
for ops, args, out in examples:
    assert run_commands(MyCalendarTwo, ops, args) == out
    for name, cls in IMPLS.items():
        assert run_with(cls, ops, args) == out, name

# edge case
check_script([(10, 20)], [True])
check_script([(10, 20), (20, 30)], [True, True])                         # kề nhau: không giao
check_script([(10, 20), (10, 20), (20, 30), (20, 30)], [True, True, True, True])
check_script([(0, 10), (0, 10), (0, 10)], [True, True, False])           # ba đoạn hệt nhau
check_script([(0, 10), (0, 10), (9, 10)], [True, True, False])           # chạm một đơn vị cuối
check_script([(0, 10), (0, 10), (10, 11)], [True, True, True])           # kề vùng trùng đôi
check_script([(0, 100), (10, 20), (30, 40), (15, 35)], [True, True, True, False])   # lồng nhau
check_script([(0, 10 ** 9), (0, 10 ** 9), (5, 6)], [True, True, False])  # biên lớn nhất
check_script([(0, 10 ** 9), (0, 1), (5, 10 ** 9)], [True, True, True])
# từ chối không làm đổi trạng thái: sau khi bị từ chối [0,10) lần ba, [10,20) và các đoạn xa vẫn nhận
check_script([(0, 10), (0, 10), (0, 10), (10, 20), (10, 20), (10, 20), (30, 40)],
             [True, True, False, True, True, False, True])
print("edge case: OK")

edge case: OK


**Phản ví dụ** cho các cách làm sai điển hình (chi tiết ở mục 13):

- Dùng đoạn **đóng**: `[0,10)` trùng đôi rồi đặt `[10,20)` bị từ chối nhầm.
- Không **hoàn tác** khi từ chối (cách mảng hiệu): sau một lần từ chối, **mọi** lần đặt sau đều bị từ chối nhầm.
- Kiểm tra với `booked` thay vì `overlaps`: từ chối ngay cả đặt trùng đôi hợp lệ (nhầm với My Calendar I).

In [24]:
# Đối chiếu nhiều kịch bản ngẫu nhiên, toạ độ nhỏ (nhiều va chạm) và toạ độ lớn (ít va chạm)
rng = random.Random(2027)

def random_bookings(rng, calls, lo, hi, max_len):
    out = []
    for _ in range(calls):
        s = rng.randint(lo, hi - 1)
        e = rng.randint(s + 1, min(s + max_len, hi))
        out.append((s, e))
    return out

count = 0
for _ in range(400):
    bookings = random_bookings(rng, rng.randint(1, 60), 0, rng.choice([12, 30, 60]), rng.choice([2, 6, 20]))
    check_script(bookings)
    count += 1
print("toạ độ nhỏ:", count, "kịch bản OK")

count = 0
for _ in range(60):
    bookings = random_bookings(rng, rng.randint(50, 150), 0, 10 ** 9, rng.choice([10 ** 7, 10 ** 8, 10 ** 9]))
    check_script(bookings)
    count += 1
print("toạ độ lớn:", count, "kịch bản OK")

toạ độ nhỏ:

 400 kịch bản OK


toạ độ lớn: 60 kịch bản OK


In [25]:
# Test lớn nhất theo ràng buộc: 1000 lệnh book
rng = random.Random(1000)
big = {
    "toạ độ nhỏ, nhiều va chạm": random_bookings(rng, 1000, 0, 200, 15),
    "toạ độ lớn ngẫu nhiên": random_bookings(rng, 1000, 0, 10 ** 9, 10 ** 8),
    "đoạn rời nhau (đều nhận)": [(10 * i, 10 * i + 5) for i in range(1000)],
    "đoạn lồng nhau tăng dần": [(i, 2000 - i) for i in range(1000)],
}
for name, bookings in big.items():
    ops, args = script(bookings)
    ref, row = None, []
    for mname, cls in IMPLS.items():
        t0 = time.perf_counter()
        got = run_with(cls, ops, args)
        dt = (time.perf_counter() - t0) * 1000
        if ref is None:
            ref = got
        assert got == ref, (name, mname)
        row.append(f"{mname} {dt:7.1f} ms")
    print(f"{name:28} nhận {sum(1 for x in ref if x is True):4}/1000 |", " | ".join(row))

toạ độ nhỏ, nhiều va chạm    nhận   83/1000 | brute     6.3 ms | two_lists     1.5 ms | diff     4.9 ms | segtree     6.2 ms
toạ độ lớn ngẫu nhiên        nhận   77/1000 | brute     5.5 ms | two_lists     1.3 ms | diff    27.6 ms | segtree     6.8 ms
đoạn rời nhau (đều nhận)     nhận 1000/1000 | brute    33.4 ms | two_lists    33.3 ms | diff    43.1 ms | segtree    18.0 ms


đoạn lồng nhau tăng dần      nhận    2/1000 | brute     0.7 ms | two_lists     0.2 ms | diff    23.5 ms | segtree     4.4 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi sự kiện là đoạn **đóng** (`<=` thay vì `<`)

**Code sai:**

```python
if max(a, s) <= min(b, e):        # đóng: hai đoạn chạm đầu mút cũng bị coi là giao
    return False
```

**Vì sao sai:** sự kiện là **nửa mở** `[s, e)`; `[0,10)` và `[10,20)` chỉ **kề nhau**, không có điểm chung. Phải dùng `<` chặt.

In [26]:
class WrongClosed:
    def __init__(self):
        self.booked, self.overlaps = [], []
    def book(self, s, e):
        for a, b in self.overlaps:
            if max(a, s) <= min(b, e):              # SAI: đóng
                return False
        for a, b in self.booked:
            lo, hi = max(a, s), min(b, e)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((s, e))
        return True

ops, args = script([(0, 10), (0, 10), (10, 20)])
print("sai :", run_with(WrongClosed, ops, args)[1:])
print("đúng:", run_with(CalTwoLists, ops, args)[1:])

sai : [True, True, False]
đúng: [True, True, True]


### Lỗi 2: Thêm sự kiện trước khi kiểm tra và **không hoàn tác**

**Code sai (cách mảng hiệu):**

```python
self._add(s, 1); self._add(e, -1)
cur = 0
for k in self.keys:
    cur += self.diff[k]
    if cur > 2:
        return False            # quên hoàn tác!
```

**Vì sao sai:** sự kiện bị từ chối đã được cộng vào mảng hiệu và **ở lại mãi**. Từ đó mảng hiệu chứa độ phủ 3 vĩnh viễn nên mọi lần đặt sau (kể cả ở vùng xa) đều bị từ chối.

In [27]:
class WrongNoRollback:
    def __init__(self):
        self.diff, self.keys = {}, []
    def _add(self, t, d):
        if t not in self.diff:
            insort(self.keys, t); self.diff[t] = 0
        self.diff[t] += d
    def book(self, s, e):
        self._add(s, 1); self._add(e, -1)
        cur = 0
        for k in self.keys:
            cur += self.diff[k]
            if cur > 2:
                return False                         # SAI: không hoàn tác
        return True

ops, args = script([(0, 10), (0, 10), (0, 10), (20, 30)])
print("sai :", run_with(WrongNoRollback, ops, args)[1:])     # lần 4 bị từ chối nhầm
print("đúng:", run_with(CalDiff, ops, args)[1:])

sai : [True, True, False, False]
đúng: [True, True, False, True]


### Lỗi 3: Kiểm tra giao với `booked` thay vì `overlaps` (nhầm với My Calendar I)

**Code sai:**

```python
for a, b in self.booked:
    if max(a, s) < min(b, e):     # giao với BẤT KỲ sự kiện nào -> từ chối
        return False
```

**Vì sao sai:** bài này **cho phép trùng đôi**; chỉ từ chối khi giao với vùng **đã trùng đôi** (`overlaps`). Kiểm tra với `booked` là luật của *My Calendar I* (729: cấm mọi chồng lấn).

In [28]:
class WrongAgainstBooked:
    def __init__(self):
        self.booked, self.overlaps = [], []
    def book(self, s, e):
        for a, b in self.booked:                      # SAI: phải là self.overlaps
            if max(a, s) < min(b, e):
                return False
        self.booked.append((s, e))
        return True

ops, args = ex1_ops, ex1_args
print("sai :", run_with(WrongAgainstBooked, ops, args)[1:])
print("đúng:", run_with(CalTwoLists, ops, args)[1:])

sai : [True, True, False, False, True, False]
đúng: [True, True, True, False, True, True]


### Lỗi 4: Ghi **cả đoạn mới** vào `overlaps` thay vì **phần giao**

**Code sai:**

```python
for a, b in self.booked:
    if max(a, s) < min(b, e):
        self.overlaps.append((s, e))      # SAI: cả đoạn mới không phải vùng trùng đôi
```

**Vì sao sai:** chỉ **phần giao** `[max(a,s), min(b,e))` bị phủ hai lần. Ghi cả `[s, e)` làm vùng trùng đôi **phình to**, nên các lần đặt sau bị từ chối nhầm.

In [29]:
class WrongWholeInterval:
    def __init__(self):
        self.booked, self.overlaps = [], []
    def book(self, s, e):
        for a, b in self.overlaps:
            if max(a, s) < min(b, e):
                return False
        for a, b in self.booked:
            if max(a, s) < min(b, e):
                self.overlaps.append((s, e))          # SAI: phải là (max(a,s), min(b,e))
        self.booked.append((s, e))
        return True

ops, args = script([(10, 20), (15, 40), (30, 35)])
print("sai :", run_with(WrongWholeInterval, ops, args)[1:])      # lần 3 bị từ chối nhầm
print("đúng:", run_with(CalTwoLists, ops, args)[1:])

sai : [True, True, False]
đúng: [True, True, True]


### Lỗi 5: Dữ liệu ở **mức lớp** làm các đối tượng dùng chung

**Code sai:**

```python
class MyCalendarTwo:
    booked = []                    # thuộc tính của LỚP, không phải của đối tượng
    overlaps = []
```

**Vì sao sai:** `booked` và `overlaps` thuộc về **lớp**, mọi đối tượng cùng thấy một list. LeetCode có thể tạo nhiều đối tượng (hoặc chạy nhiều test trong cùng tiến trình) nên dữ liệu của test trước **rò** sang test sau. Hãy khởi tạo trong `__init__` bằng `self.booked = []`.

In [30]:
class WrongClassLevel:
    booked = []                                       # SAI: dùng chung
    overlaps = []
    def book(self, s, e):
        for a, b in self.overlaps:
            if max(a, s) < min(b, e):
                return False
        for a, b in self.booked:
            lo, hi = max(a, s), min(b, e)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((s, e))
        return True

x, y = WrongClassLevel(), WrongClassLevel()
print(x.book(0, 10), x.book(0, 10))
print("đối tượng y mới tạo, book(0, 10) ->", y.book(0, 10), "(đúng phải là True vì y còn trống)")
a, b = CalTwoLists(), CalTwoLists()
print(a.book(0, 10), a.book(0, 10), "|", b.book(0, 10))

True True
đối tượng y mới tạo, book(0, 10) -> False (đúng phải là True vì y còn trống)
True True | True


## 14. Mẹo & nhận dạng pattern

- **"Không cho quá `k` lần chồng" trên đoạn thời gian** (lịch, phòng họp, tàu): nghĩ tới **độ phủ** (mỗi điểm bị bao nhiêu đoạn phủ). Hai công cụ chuẩn: **mảng hiệu** (`+1` ở đầu, `−1` ở cuối) và **cây phân đoạn** (cộng khoảng, max khoảng).
- **Nửa mở `[s, e)`:** hai đoạn kề nhau không giao; công thức giao là `max(đầu) < min(cuối)` (chặt). Nhớ ghi `−1` tại `e` (không phải `e + 1`) khi dùng mảng hiệu.
- **Thiết kế lớp:** trạng thái đặt trong `__init__` qua `self`; kiểm tra **trước**, ghi **sau** (hoặc hoàn tác cẩn thận). Một thao tác bị từ chối **không được thay đổi trạng thái**.
- **Họ bài My Calendar:** I (cấm chồng, kiểm bằng danh sách hoặc `bisect`), II (cấm trùng ba), III (trả độ phủ lớn nhất) — cùng một họ mảng hiệu / cây phân đoạn.
- **Chọn cấu trúc theo ràng buộc:** `n ≤ 1000` cho phép `O(n)` mỗi lần; `n` lớn hơn (≥ 10^5) cần cây phân đoạn hoặc cây cân bằng.
- **Test thiết kế lớp:** viết một **oracle ngây thơ** (mảng độ phủ từng đơn vị trên toạ độ nhỏ) và so sánh trên kịch bản ngẫu nhiên có seed; viết sẵn bộ chạy chuỗi lệnh để dùng chung cho mọi cài đặt.

## 15. Biến thể

1. **My Calendar I (729):** cấm **mọi** chồng lấn (tức `k = 1`).
2. **My Calendar III (732):** sau mỗi `book`, trả **độ phủ lớn nhất** (số sự kiện chồng nhiều nhất) — mảng hiệu duyệt toàn bộ hoặc cây phân đoạn.
3. **Cho phép `k` lần chồng** (tổng quát hoá): `MyCalendarK(k)` — chỉ cần đổi hằng `2` thành `k`.
4. **Meeting Rooms II (253):** số phòng tối thiểu = độ phủ lớn nhất.
5. **Range Module (715), Count Integers in Intervals (2276):** quản lý hợp các khoảng.
6. **Dùng cây cân bằng / `SortedList`:** thay `list + insort` bằng cấu trúc có chèn `O(log n)` cho `n` rất lớn.

Dưới đây là biến thể 3 (tổng quát `k`) và 2 (độ phủ lớn nhất), kiểm tra bằng oracle:

In [31]:
class MyCalendarK:
    def __init__(self, k):
        self.k, self.diff, self.keys = k, {}, []

    def _add(self, t, d):
        if t not in self.diff:
            insort(self.keys, t)
            self.diff[t] = 0
        self.diff[t] += d

    def book(self, s, e):
        self._add(s, 1); self._add(e, -1)
        cur = 0
        for key in self.keys:
            cur += self.diff[key]
            if cur > self.k:
                self._add(s, -1); self._add(e, 1)
                return False
        return True

class MyCalendarThree:
    def __init__(self):
        self.diff = {}

    def book(self, s, e):
        self.diff[s] = self.diff.get(s, 0) + 1
        self.diff[e] = self.diff.get(e, 0) - 1
        cur = best = 0
        for t in sorted(self.diff):
            cur += self.diff[t]
            best = max(best, cur)
        return best

rng = random.Random(732)
for _ in range(200):
    bookings = random_bookings(rng, rng.randint(1, 30), 0, 40, 10)
    ops, args = script(bookings)
    assert run_with(lambda: MyCalendarK(2), ops, args) == run_with(CalTwoLists, ops, args)
    # oracle cho bài III: mảng độ phủ theo đơn vị
    cnt, expected = [0] * 64, []
    for s, e in bookings:
        for x in range(s, e):
            cnt[x] += 1
        expected.append(max(cnt))
    three = MyCalendarThree()
    assert [three.book(s, e) for s, e in bookings] == expected
print("MyCalendarK(2) == MyCalendarTwo và MyCalendarThree khớp oracle: OK")
print(run_with(lambda: MyCalendarK(1), *script([(10, 20), (15, 25), (20, 30)])))   # My Calendar I

MyCalendarK(2) == MyCalendarTwo và MyCalendarThree khớp oracle: OK
[None, True, False, True]


## 16. So sánh các cách

| Cách | Ý tưởng | Time mỗi `book` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm tại đầu mút | Với mỗi ứng viên điểm, đếm số sự kiện phủ nó | `O(n²)` | `O(n)` | Oracle; hiểu bản chất "độ phủ" |
| 2. `booked` + `overlaps` | Lưu vùng trùng đôi; từ chối nếu giao nó | `O(n)` | `O(n)` | **Cách nộp chuẩn** cho `n ≤ 1000` |
| 3. Mảng hiệu thưa | Hàm độ phủ qua `+1/−1`, cộng dồn, hoàn tác khi vi phạm | `O(n)` | `O(n)` | Ngắn gọn; mở rộng sang My Calendar III |
| 4. Cây phân đoạn động | Cộng `+1` khoảng, hỏi max khoảng | `O(log C)` | `O(n log C)` | `n` rất lớn, toạ độ lớn |

Kết luận: nộp **cách 2**. Cách 3 là lựa chọn gọn nhất khi cần mở rộng sang bài khác; cách 4 là lời giải "đúng nghĩa" cho dữ liệu lớn.

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm/lớp rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh.

In [32]:
print(max(5, 10), min(15, 20))
print(intersect((10, 40), (5, 15)), intersect((0, 10), (10, 20)))
print(run_with(CalTwoLists, *script([(1, 5), (1, 5), (4, 8), (5, 8)])))
print(sorted({3: 1, 1: 2, 2: 0}))

10 15
(10, 15) None
[None, True, True, False, True]
[1, 2, 3]


### Bài 2 (tính tay)

Với dãy đặt lịch `(1, 5), (3, 7), (4, 6), (6, 9)`, hãy tính tay kết quả `True/False` của từng lần và danh sách `overlaps` cuối cùng của cách 2. Gán vào `my_results` (list 4 giá trị bool) và `my_overlaps` (list các tuple theo thứ tự được thêm). Hiện để `None`.

In [33]:
my_results = None
my_overlaps = None
# Sau khi điền, bỏ comment để kiểm tra:
# cal = CalTwoLists()
# truth = [cal.book(s, e) for s, e in [(1, 5), (3, 7), (4, 6), (6, 9)]]
# assert my_results == truth and my_overlaps == cal.overlaps

### Bài 3 (điền TODO)

Viết hai hàm: `inter(a, b)` trả phần giao của hai nửa khoảng (tuple) hoặc `None` nếu không giao; và `can_book(overlaps, s, e)` trả `True` nếu `[s, e)` **không giao** với đoạn nào trong `overlaps`.

In [34]:
def inter(a, b):
    # TODO
    pass

def can_book(overlaps, s, e):
    # TODO
    pass

# Kiểm tra (bỏ comment sau khi viết xong):
# assert inter((10, 20), (15, 30)) == (15, 20)
# assert inter((10, 20), (20, 30)) is None
# assert can_book([(10, 20)], 5, 10) is True
# assert can_book([(10, 20)], 5, 11) is False
# assert can_book([], 0, 10 ** 9) is True

### Bài 4 (tự viết lại)

Viết lớp `MyCalendarOne` (bài 729): `book(start, end)` trả `True` và lưu nếu `[start, end)` **không giao với bất kỳ** sự kiện nào đã có, ngược lại trả `False`. Không xem lại code ở trên.

In [35]:
class MyCalendarOne:
    # TODO: __init__ và book
    pass

# ops = ["MyCalendarOne", "book", "book", "book"]
# args = [[], [10, 20], [15, 25], [20, 30]]
# assert run_commands(MyCalendarOne, ops, args) == [None, True, False, True]

### Bài 5 (biến thể / mini challenge)

Viết lớp `MyCalendarThree` (bài 732): `book(start, end)` luôn lưu sự kiện và trả **số sự kiện chồng nhiều nhất tại một thời điểm** sau khi thêm. Kiểm tra bằng oracle độc lập dưới đây.

In [36]:
class MyCalendarThree:
    # TODO: __init__ và book
    pass

def three_oracle(bookings):
    cnt, out = [0] * 64, []
    for s, e in bookings:
        for x in range(s, e):
            cnt[x] += 1
        out.append(max(cnt))
    return out

# bookings = [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]
# t = MyCalendarThree()
# assert [t.book(s, e) for s, e in bookings] == three_oracle(bookings) == [1, 1, 2, 3, 3, 3]

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `max(5,10)` là số lớn hơn, `min(15,20)` số nhỏ hơn. `intersect` trả `None` khi hai đoạn chỉ kề nhau.
- Mức 2: lịch `(1,5),(1,5),(4,8),(5,8)`: hai lần đầu trùng đôi ở `[1,5)`; `(4,8)` giao `[1,5)` tại `[4,5)`.
- Mức 3: kết quả theo thứ tự: `10 15`, `(10, 15) None`, `[None, True, True, False, True]`, `[1, 2, 3]`.

**Bài 2.**
- Mức 1: lần 1 luôn `True`. Lần 2: `(3,7)` giao `(1,5)` tại `[3,5)`, thêm vào `overlaps`.
- Mức 2: lần 3 `(4,6)` giao `overlaps = [(3,5)]` tại `[4,5)` → `False`, không ghi.
- Mức 3: lần 4 `(6,9)`: không giao `(3,5)`; giao `(3,7)` tại `[6,7)` thêm vào `overlaps`; giao `(1,5)` rỗng. Kết quả `[True, True, False, True]`, `overlaps = [(3, 5), (6, 7)]`.

**Bài 3.**
- Mức 1: `lo = max(a[0], b[0])`, `hi = min(a[1], b[1])`.
- Mức 2: có giao khi `lo < hi` (chặt).
- Mức 3: `can_book` = `not any(inter(o, (s, e)) for o in overlaps)`.

**Bài 4.**
- Mức 1: `self.events = []` trong `__init__`.
- Mức 2: trong `book`, duyệt `self.events`, từ chối nếu `max(a, s) < min(b, e)`.
- Mức 3: nếu duyệt xong không có giao thì `append((s, e))` và `return True`.

**Bài 5.**
- Mức 1: dùng mảng hiệu `dict`: `+1` tại `s`, `−1` tại `e`.
- Mức 2: sau mỗi `book`, duyệt các mốc theo `sorted(...)`, cộng dồn.
- Mức 3: trả `max` của giá trị cộng dồn (nhớ khởi tạo `best = 0`).

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
assert (max(5, 10), min(15, 20)) == (10, 15)
assert intersect((10, 40), (5, 15)) == (10, 15) and intersect((0, 10), (10, 20)) is None
assert run_with(CalTwoLists, *script([(1, 5), (1, 5), (4, 8), (5, 8)])) == [None, True, True, False, True]
assert sorted({3: 1, 1: 2, 2: 0}) == [1, 2, 3]
print("Bài 1: 10 15 | (10, 15) None | [None, True, True, False, True] | [1, 2, 3]")

# Bài 2
my_results = [True, True, False, True]
my_overlaps = [(3, 5), (6, 7)]
cal = CalTwoLists()
truth = [cal.book(s, e) for s, e in [(1, 5), (3, 7), (4, 6), (6, 9)]]
assert my_results == truth and my_overlaps == cal.overlaps
print("Bài 2: OK", my_results, my_overlaps)

# Bài 3
def inter(a, b):
    lo, hi = max(a[0], b[0]), min(a[1], b[1])
    return (lo, hi) if lo < hi else None

def can_book(overlaps, s, e):
    return not any(inter(o, (s, e)) for o in overlaps)

assert inter((10, 20), (15, 30)) == (15, 20)
assert inter((10, 20), (20, 30)) is None
assert can_book([(10, 20)], 5, 10) is True
assert can_book([(10, 20)], 5, 11) is False
assert can_book([], 0, 10 ** 9) is True
print("Bài 3: OK")

Bài 1: 10 15 | (10, 15) None | [None, True, True, False, True] | [1, 2, 3]
Bài 2: OK [True, True, False, True] [(3, 5), (6, 7)]
Bài 3: OK


In [38]:
# Bài 4
class MyCalendarOne:
    def __init__(self):
        self.events = []

    def book(self, start, end):
        for a, b in self.events:
            if max(a, start) < min(b, end):
                return False
        self.events.append((start, end))
        return True

ops = ["MyCalendarOne", "book", "book", "book"]
args = [[], [10, 20], [15, 25], [20, 30]]
assert run_commands(MyCalendarOne, ops, args) == [None, True, False, True]
rng = random.Random(4)
for _ in range(200):
    bookings = random_bookings(rng, rng.randint(1, 30), 0, 40, 8)
    cnt, expected = [0] * 64, []
    for s, e in bookings:
        if max(cnt[s:e]) >= 1:
            expected.append(False)
        else:
            for x in range(s, e):
                cnt[x] += 1
            expected.append(True)
    o, a = script(bookings)
    assert run_with(MyCalendarOne, o, a) == [None] + expected
print("Bài 4: OK")

# Bài 5
class MyCalendarThree:
    def __init__(self):
        self.diff = {}

    def book(self, start, end):
        self.diff[start] = self.diff.get(start, 0) + 1
        self.diff[end] = self.diff.get(end, 0) - 1
        cur = best = 0
        for t in sorted(self.diff):
            cur += self.diff[t]
            best = max(best, cur)
        return best

def three_oracle(bookings):
    cnt, out = [0] * 64, []
    for s, e in bookings:
        for x in range(s, e):
            cnt[x] += 1
        out.append(max(cnt))
    return out

bookings = [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]
t = MyCalendarThree()
assert [t.book(s, e) for s, e in bookings] == three_oracle(bookings) == [1, 1, 2, 3, 3, 3]
rng = random.Random(5)
for _ in range(200):
    bookings = random_bookings(rng, rng.randint(1, 30), 0, 40, 10)
    t = MyCalendarThree()
    assert [t.book(s, e) for s, e in bookings] == three_oracle(bookings)
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "lịch cho phép trùng đôi" (hai danh sách):**

```python
class MyCalendarTwo:

    def __init__(self):
        self.booked = []        # mọi sự kiện đã nhận
        self.overlaps = []      # các đoạn đã bị phủ đúng 2 lần

    def book(self, startTime: int, endTime: int) -> bool:
        for a, b in self.overlaps:                       # 1. giao vùng trùng đôi => từ chối
            if max(a, startTime) < min(b, endTime):
                return False
        for a, b in self.booked:                         # 2. phần giao mới => vùng trùng đôi mới
            lo, hi = max(a, startTime), min(b, endTime)
            if lo < hi:
                self.overlaps.append((lo, hi))
        self.booked.append((startTime, endTime))         # 3. ghi sự kiện (chỉ khi hợp lệ)
        return True
```

**Ghi nhớ một câu:** *"Giao với vùng đã trùng đôi thì từ chối; nếu nhận thì phần giao với từng sự kiện cũ thành vùng trùng đôi mới."*

**Checklist trước khi nộp:**

- [ ] Trạng thái khởi tạo trong `__init__` bằng `self....`, **không** để ở mức lớp.
- [ ] Công thức giao dùng `max(đầu) < min(cuối)` (chặt: nửa mở, kề nhau không giao).
- [ ] Kiểm tra với `overlaps` (không phải `booked`).
- [ ] Phần giao `(max(a, s), min(b, e))` mới được thêm vào `overlaps`, không phải cả `(s, e)`.
- [ ] Sự kiện bị từ chối **không** thay đổi trạng thái (kiểm tra trước, ghi sau; hoặc hoàn tác).
- [ ] Trả `bool` đúng kiểu (`True`/`False`).
- [ ] Đã thử: hai đoạn kề nhau, ba đoạn trùng hệt nhau, đoạn lồng nhau, `[0, 10^9)`, và sau khi từ chối vẫn nhận được các đoạn xa.